In [2]:
import astroquery
import astropy
import pyvo

print("astroquery :", astroquery.__version__)
print("astropy    :", astropy.__version__)
print("pyvo       :", pyvo.__version__)

astroquery : 0.4.11
astropy    : 6.1.7
pyvo       : 1.8.1


In [3]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

planets = NasaExoplanetArchive.query_criteria(
    table="ps",
    select="*",
    where="default_flag=1 and disc_facility like '%TESS%'"
)

print(planets.colnames)

['pl_name', 'pl_letter', 'hostname', 'hd_name', 'hip_name', 'tic_id', 'gaia_dr2_id', 'gaia_dr3_id', 'default_flag', 'pl_refname', 'sy_refname', 'disc_pubdate', 'disc_year', 'discoverymethod', 'disc_locale', 'disc_facility', 'disc_instrument', 'disc_telescope', 'disc_refname', 'ra', 'rastr', 'dec', 'decstr', 'glon', 'glat', 'elon', 'elat', 'pl_orbper', 'pl_orbpererr1', 'pl_orbpererr2', 'pl_orbperlim', 'pl_orbperstr', 'pl_orblpererr1', 'pl_orblper', 'pl_orblpererr2', 'pl_orblperlim', 'pl_orblperstr', 'pl_orbsmax', 'pl_orbsmaxerr1', 'pl_orbsmaxerr2', 'pl_orbsmaxlim', 'pl_orbsmaxstr', 'pl_orbincl', 'pl_orbinclerr1', 'pl_orbinclerr2', 'pl_orbincllim', 'pl_orbinclstr', 'pl_orbtper', 'pl_orbtpererr1', 'pl_orbtpererr2', 'pl_orbtperlim', 'pl_orbtperstr', 'pl_orbeccen', 'pl_orbeccenerr1', 'pl_orbeccenerr2', 'pl_orbeccenlim', 'pl_orbeccenstr', 'pl_eqt', 'pl_eqterr1', 'pl_eqterr2', 'pl_eqtlim', 'pl_eqtstr', 'pl_occdep', 'pl_occdeperr1', 'pl_occdeperr2', 'pl_occdeplim', 'pl_occdepstr', 'pl_insol', 

In [4]:
pip install dace-query


Note: you may need to restart the kernel to use updated packages.


In [5]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive
from dace_query.spectroscopy import Spectroscopy
import pandas as pd

print("Downloading confirmed TESS planets...")

planets = NasaExoplanetArchive.query_criteria(
    table="ps",
    select="hostname,pl_name,tic_id,gaia_dr3_id",
    where="default_flag=1 and disc_facility like '%TESS%'"
)

rows = []

print(f"Checking {len(planets)} planets...\n")

for row in planets[:2]:

    host = str(row["hostname"]).strip()

    try:
        # Query DACE for RV time-series
        ts = Spectroscopy.get_timeseries(
            target=host,
            sorted_by_instrument=True
        )

        if len(ts) == 0:
            continue

        for instrument in ts.keys():

            nrv = 0

            for drs in ts[instrument]:

                for mode in ts[instrument][drs]:

                    try:
                        nrv += len(
                            ts[instrument][drs][mode]["rv"]
                        )
                    except Exception:
                        pass

            rows.append(
                {
                    "tic_id": row["tic_id"],
                    "hostname": host,
                    "planet": row["pl_name"],
                    "gaia_dr3_id": row["gaia_dr3_id"],
                    "instrument": instrument,
                    "n_rv": nrv,
                    "dace_page":
                    f"https://dace.unige.ch/spectroscopy/?target={host.replace(' ','%20')}"
                }
            )

            print(host, instrument, nrv)

    except Exception:
        pass

df = pd.DataFrame(rows)

df.to_csv(
    "tess_planets_with_rv.csv",
    index=False
)

print("\nDone.")
print(df.head())

2026-06-29 18:35:44,216 - WARNING - File .dacerc not found. You are requesting data in public mode. To change this behaviour, create a .dacerc file in your home directory and fill it with your API key. More infos on https://dace.unige.ch


Checking 897 planets...

HD 39091 CORAVEL 5
HD 39091 UCLES 70
HD 39091 CORALIE98 29
HD 39091 HARPS03 129
HD 39091 CORALIE07 24
HD 39091 HARPS15 430
HD 39091 CORALIE14 94
HD 39091 ESPRESSO18 555
LHS 3844 ESPRESSO18 2
LHS 3844 ESPRESSO19 69

Done.
          tic_id  hostname    planet                   gaia_dr3_id instrument  \
0  TIC 261136679  HD 39091  pi Men c  Gaia DR3 4623036865373793408    CORAVEL   
1  TIC 261136679  HD 39091  pi Men c  Gaia DR3 4623036865373793408      UCLES   
2  TIC 261136679  HD 39091  pi Men c  Gaia DR3 4623036865373793408  CORALIE98   
3  TIC 261136679  HD 39091  pi Men c  Gaia DR3 4623036865373793408    HARPS03   
4  TIC 261136679  HD 39091  pi Men c  Gaia DR3 4623036865373793408  CORALIE07   

   n_rv                                          dace_page  
0     5  https://dace.unige.ch/spectroscopy/?target=HD%...  
1    70  https://dace.unige.ch/spectroscopy/?target=HD%...  
2    29  https://dace.unige.ch/spectroscopy/?target=HD%...  
3   129  https://dace.u

In [ ]:
df.to_csv('abc.csv')

,tic_id,hostname,planet,gaia_dr3_id,instrument,n_rv,dace_page
0,TIC 261136679,HD 39091,pi Men c,Gaia DR3 4623036865373793408,CORAVEL,5,https://dace.unige.ch/spectroscopy/?target=HD%...
1,TIC 261136679,HD 39091,pi Men c,Gaia DR3 4623036865373793408,UCLES,70,https://dace.unige.ch/spectroscopy/?target=HD%...
2,TIC 261136679,HD 39091,pi Men c,Gaia DR3 4623036865373793408,CORALIE98,29,https://dace.unige.ch/spectroscopy/?target=HD%...
3,TIC 261136679,HD 39091,pi Men c,Gaia DR3 4623036865373793408,HARPS03,129,https://dace.unige.ch/spectroscopy/?target=HD%...
4,TIC 261136679,HD 39091,pi Men c,Gaia DR3 4623036865373793408,CORALIE07,24,https://dace.unige.ch/spectroscopy/?target=HD%...
5,TIC 261136679,HD 39091,pi Men c,Gaia DR3 4623036865373793408,HARPS15,430,https://dace.unige.ch/spectroscopy/?target=HD%...
6,TIC 261136679,HD 39091,pi Men c,Gaia DR3 4623036865373793408,CORALIE14,94,https://dace.unige.ch/spectroscopy/?target=HD%...
7,TIC 261136679,HD 39091,pi Men c,Gaia DR3 4623036865373793408,ESPRESSO18,555,https://dace.unige.ch/spectroscopy/?target=HD%...
8,TIC 410153553,LHS 3844,LHS 3844 b,Gaia DR3 6385548541499112448,ESPRESSO18,2,https://dace.unige.ch/spectroscopy/?target=LHS...
9,TIC 410153553,LHS 3844,LHS 3844 b,Gaia DR3 6385548541499112448,ESPRESSO19,69,https://dace.unige.ch/spectroscopy/?target=LHS...
